# Day 2 — ILT 1: CDC Concepts — WAL and Logical Replication from Supabase
**Time:** Day 2, Morning
**What comes next:** ILT 2 — Lakeflow Connect & Storage Credentials, then HOL 1 — hands-on Lakeflow Connect setup

---
### What we cover today
1. Intro to ingestion patterns — what ingestion is, Full vs Incremental Load, batch vs streaming, common ingestion issues
2. What is CDC and why we need it (not just incremental load)
3. How PostgreSQL WAL works — the general mechanism behind log-based CDC
4. Logical replication in Supabase — enabling it
5. Connecting Databricks to Supabase via JDBC (to see the mechanics)
6. Reading tables from Supabase directly into Spark

> **Instructor note:** 60 minutes. ~10 min ingestion-patterns theory (what is ingestion, full vs incremental, batch vs streaming — parking streaming, common issues), ~10 min on CDC theory (draw on board: WAL → replication slot → one of the two Lakeflow Connect modes), ~20 min JDBC connection live demo (to show what's happening under the hood), ~15 min WAL/logical replication setup in Supabase UI, ~5 min wrap-up. **Important framing:** WAL/logical replication is general Postgres CDC theory, and it's what powers Lakeflow Connect's *log-based* connector mode — but GlobalMart's real, running `orders`/`order_items` pipeline is configured in Lakeflow Connect's *other* mode, query/cursor-based (cursor column `updated_at`), which does **not** stream the WAL and **cannot** catch hard deletes. ILT 2 covers this distinction in full. Today's session builds the WAL/CDC mental model as general-purpose CDC theory — not as a description of what GlobalMart's actual pipeline does.

## Section 0 — Intro to Ingestion Patterns

### What is ingestion?

**Ingestion** is the act of bringing data from a source system into the lakehouse — before any cleaning, joining, or modeling happens. It's the very first hop: source → Bronze.

Every CSV you've read with `spark.read.csv(...)` since Day 1 was already ingestion — just the simplest possible form: a one-time full file drop.

Every ingestion pattern makes 2 decisions:
1. **How much** data to bring each run — **Full Load** or **Incremental Load**?
2. **How continuously** — **Batch** (scheduled runs) or **Streaming** (continuous, row-by-row)?

### Decision 1 — Full Load vs Incremental Load

| | Full Load | Incremental Load |
|---|---|---|
| **What it reads** | The entire source, every run | Only new/changed rows since last run |
| **How** | Overwrite the whole table | A watermark filter, e.g. `WHERE created_at > last_run` |
| **Cost** | Wasteful at scale, but simple & safe | Cheap and fast, but only as good as the filter |
| **Good for** | Small reference tables (`payment_methods`, `shipping_tier`) | Tables that only ever grow (new orders, new signups) |

> A quick, honest preview: a plain incremental watermark only catches new **inserts**. When a source needs every **UPDATE** and **DELETE** captured too, there's a stronger pattern called **CDC (Change Data Capture)** — that's the rest of this session.

### Decision 2 — Batch focus, streaming parked

- **Batch** — runs on a schedule or a trigger, reads a chunk of data, processes it, stops. Simple to reason about, easy to re-run, easy to debug. **Every ingestion pattern we build hands-on in this bootcamp is batch** — including today's Lakeflow Connect pipeline and Day 3/4's Autoloader.
- **Streaming** — processes each event continuously, as it happens, with sub-second latency, but far more operational complexity (always-on compute, checkpointing, backpressure). We'll mention it again when relevant (Structured Streaming touches Week 3's ops/governance material), but we won't build a streaming pipeline in this course. Parked, not ignored.

### Common ingestion issues you'll hit in the real world

| Issue | What Happens |
|---|---|
| **Schema evolution** | Source adds, removes, or renames a column mid-course |
| **Duplicates** | The same row lands twice — a retried job, an overlapping incremental window |
| **Late-arriving data** | A row logically "for yesterday" only shows up in today's batch |
| **Format mismatches** | CSV vs JSON vs Parquet, inconsistent date formats, encoding issues |

This is exactly why Bronze stays raw and untouched — these issues get handled deliberately in Silver, not silently patched over during ingestion. Full preview of that cleanup logic is Day 5.

### Bridge — GlobalMart's two ingestion sources

- **Postgres (Supabase)** — a live transactional database (`orders`, `order_items`). Rows are inserted, updated, and deleted continuously. Needs a **database-aware** approach → CDC / Lakeflow Connect (the rest of today, plus ILT 2).
- **ADLS (file drops)** — `customers`, `products`, `addresses`, `payments`, and more, landing as files periodically. Needs a **file-aware** approach → Autoloader (continues Day 3/4).

Same goal (get data into Bronze), two different tools, because the sources behave completely differently. Today's deep dive is the Postgres half.

## Section 1 — Why CDC? Isn't Incremental Load Enough?

As we just saw: **Incremental Load** reads only new rows using a watermark (date filter).  
It works for **inserts only**. But what about **updates and deletes**?

### The Problem with Incremental Load for Updates

```
customers table in Supabase:

Day 1:  CustomerID=C001, email=raj@old.com,   city=Mumbai    ← loaded to Bronze
Day 5:  CustomerID=C001, email=raj@new.com,   city=Delhi     ← customer UPDATED

Incremental Load with date filter:
  filter: WHERE created_at > last_run
  → C001 was created before last_run, so it is SKIPPED
  → Bronze still has raj@old.com and Mumbai — WRONG!
```

**CDC captures the UPDATE** — it logs that C001 changed from Mumbai to Delhi.  
That change gets propagated to Bronze → Silver → Gold automatically.

### The 3 change types CDC captures

| Operation | Example | Result in Lakehouse |
|-----------|---------|--------------------|
| **INSERT** | New customer signs up | New row added |
| **UPDATE** | Customer changes city | Existing row updated (or history tracked) |
| **DELETE** | Customer deletes account | Row removed (or marked deleted) |

## Section 2 — PostgreSQL WAL (Write-Ahead Log)

### What is WAL?

**WAL = Write-Ahead Log** — a file where PostgreSQL records every change BEFORE it commits it to the actual table.

Think of it like a bank's transaction journal:
```
Bank transaction journal:
  10:00 — Raj transferred ₹1000 to Priya
  10:01 — Account balance updated: Raj -₹1000, Priya +₹1000
  10:02 — New customer opened account

PostgreSQL WAL:
  10:00 — INSERT into customers: {id=C001, name=Raj, city=Mumbai}
  10:01 — UPDATE customers: C001 city changed Mumbai → Delhi
  10:02 — DELETE from customers: C999 removed
```

**WAL's original purpose:** crash recovery — if Postgres crashes, it replays the WAL to restore data.  
**CDC uses WAL:** we read the WAL stream instead of querying the table, so we capture EVERY change.

### Logical Replication — making WAL readable

By default, WAL is in a binary format only Postgres understands.  
**Logical replication** decodes WAL into human-readable change events (INSERT/UPDATE/DELETE).

```
WAL (binary):      0x4F02A1B4...
After decoding:    {"op": "UPDATE", "table": "customers", "id": "C001", "city": {"old": "Mumbai", "new": "Delhi"}}
```

### How to enable Logical Replication in Supabase

Supabase makes this easy through the Dashboard:

```
Supabase Dashboard
  → Database
  → Replication
  → Enable Row Level Changes for the tables you want to track
  (This sets wal_level = logical in PostgreSQL config)
```

> **We will do this in the hands-on (2:00 PM).**  
> For now — understand that enabling this is a one-time setup in Supabase UI.

## Section 3 — CDC Architecture for GlobalMart

### Full CDC Pipeline (general pattern)

```
Supabase PostgreSQL
       |
   WAL stream (logical replication)
       |
       v
  [Option A: Lakeflow Connect, log-based mode]   ← Databricks-managed, reads WAL automatically
  [Option B: Debezium → Kafka]                    ← Production-grade, self-managed, complex to operate
  [Option C: Direct JDBC polling]                  ← Manual, for learning the mechanics (this session + HOL 2)
       |
       v
  Bronze (Delta)  — raw CDC events or latest snapshot
       |
       v
  Silver — MERGE new/changed rows into clean table
       |
       v
  Gold  — aggregations always reflect latest customer data
```

### What GlobalMart Actually Uses

**Lakeflow Connect** is the production tool for `orders` and `order_items` — but it's important to be precise about *which* connector mode. Lakeflow Connect can run **log-based** (reads the WAL directly, catches every DELETE) or **query/cursor-based** (re-queries `WHERE updated_at > last_seen_value` on a schedule, misses hard deletes). **GlobalMart's real, running pipeline (`orders_data_ingestion_cdc`) uses query/cursor-based mode** — cursor column `updated_at`, history tracking Off (SCD1). No JDBC code, no manual slot management either way — you configure a connection once (**ILT 2** walks through the real setup end to end) and point a pipeline at the tables you want.

Today's JDBC + WAL walkthrough below is deliberately manual — it exists so you understand the general log-based CDC mechanism (the *other* mode Lakeflow Connect supports, but not the one configured for GlobalMart). **HOL 2** repeats this JDBC/WAL exercise hands-on for the same reason: seeing the raw mechanics builds the intuition for how log-based CDC works, even though GlobalMart's actual pipeline takes the cursor-based path instead.

| Approach | Captures Inserts | Captures Updates | Captures Deletes | Complexity | Used for |
|----------|-----------------|-----------------|-----------------|------------|----------|
| Full Load | Yes (all) | Yes (all) | Yes (by full replace) | Low | Small reference tables |
| Incremental (`created_at`) | Yes | No | No | Low | Append-only tables |
| JDBC + `updated_at` watermark | Yes | Yes | No | Low-Medium | Teaching pattern only |
| Direct JDBC + WAL replication slot | Yes | Yes | Yes | Medium | Teaching pattern (HOL 2) — log-based mode, general CDC theory |
| **Lakeflow Connect — query/cursor mode (managed)** | **Yes** | **Yes** | **No** | **Low (managed)** | **GlobalMart production — `orders`, `order_items` (ILT 2 / HOL 1)** |
| Lakeflow Connect — log-based mode (managed) | Yes | Yes | Yes | Low (managed) | Available, but not what GlobalMart's real pipeline uses |
| Debezium + Kafka (self-managed) | Yes | Yes | Yes | High | Not used in this bootcamp |

## Setup — ADLS + Supabase Credentials

In [ ]:
# ─── Supabase (PostgreSQL) Credentials ───────────────────────────────────────
# WARNING: Do NOT commit this notebook to GitHub with the real password filled in
# In production, use Databricks Secrets: dbutils.secrets.get(scope, key)
# (This session teaches the raw JDBC mechanics by hand. ILT 2 replaces this
#  entirely with a governed Unity Catalog Connection + Lakeflow Connect
#  pipeline — no JDBC code, no password in a notebook cell.)

SUPABASE_HOST     = "aws-0-ap-south-1.pooler.supabase.com"
SUPABASE_PORT     = "5432"
SUPABASE_DB       = "postgres"
SUPABASE_USER     = "postgres.isqcnhvlfnjszllicxqi"
SUPABASE_PASSWORD = "YOUR_SUPABASE_PASSWORD"  # ← paste from manager's credentials

# JDBC URL — standard PostgreSQL format
jdbc_url = f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

# Connection properties — always include ssl=true for Supabase
connection_properties = {
    "user"     : SUPABASE_USER,
    "password" : SUPABASE_PASSWORD,
    "driver"   : "org.postgresql.Driver",
    "ssl"      : "true",
    "sslmode"  : "require"
}

# ─── Bronze destination — a Unity Catalog table, not a raw ADLS path ─────────
# GlobalMart's real Bronze/Silver/Gold pipeline lives in the `gbmart` catalog.
# We only need the catalog + schema name here — no storage account, container,
# or key. Unity Catalog manages where the data physically lands.
GBMART_CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GBMART_CATALOG}.bronze")

print("JDBC URL:", jdbc_url)
print("Bronze target catalog.schema:", f"{GBMART_CATALOG}.bronze")
print("Credentials set — ready to connect to Supabase")

## Section 4 — Reading from Supabase via JDBC

In [ ]:
# ─── Read customers table directly from Supabase PostgreSQL ──────────────────
# This is a DIRECT database read — no CSV file needed!
# Spark connects to Supabase, runs a SELECT, and returns a DataFrame

# NOTE: The PostgreSQL JDBC driver must be installed on your Databricks cluster
# Go to: Cluster → Libraries → Install New → Maven → search 'postgresql'
# Package: org.postgresql:postgresql:42.6.0

try:
    customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = "customers",   # table name in Supabase
            properties = connection_properties
        )

    print(f"Rows read from Supabase customers: {customers_df.count():,}")
    print("\nSchema from Supabase:")
    customers_df.printSchema()
    customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting steps:")
    print("  1. Make sure the PostgreSQL JDBC driver is installed on the cluster")
    print("  2. Check that the password is correct")
    print("  3. Try: Cluster → Libraries → Install: org.postgresql:postgresql:42.6.0")

In [ ]:
# ─── Read with a SQL query instead of full table ──────────────────────────────
# You can push SQL down to Postgres — only read what you need
# Wrap the query in parentheses and alias it as 'query'

# This reads only customers updated in the last 30 days
# This is the CDC/incremental pattern — read only changed rows
query = """
    (SELECT *
     FROM customers
     WHERE updated_at > NOW() - INTERVAL '30 days'
     ORDER BY updated_at DESC) AS recent_customers
"""

try:
    recent_customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = query,
            properties = connection_properties
        )

    print(f"Customers updated in last 30 days: {recent_customers_df.count():,}")
    recent_customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Query failed: {e}")

In [ ]:
# ─── Simulate CDC: read all 8 tables from Supabase and land in Bronze ─────────
# In production, this is what an ingestion pipeline does under the hood:
#   1. Connect to Supabase via JDBC
#   2. Read each table (full load or watermark filter)
#   3. Save to Bronze as a governed Unity Catalog managed table
#      (gbmart.bronze.<table> — the same destination the real Lakeflow
#      Connect pipeline in ILT 2 lands into; we do it by hand here so the
#      mechanics aren't a black box)
#
# NOTE: only orders/order_items are genuinely Postgres-sourced in the real
# pipeline — the rest of this list is included here purely so this JDBC demo
# has more tables to loop over. In the real GlobalMart build, customers/
# products/addresses/payments/payment_methods arrive via ADLS Autoloader,
# not Postgres (see Day 3/4).

tables_to_ingest = [
    "customers",
    "orders",
    "order_items",
    "products",
    "payments",
    "addresses",
    "returns",
    "payment_methods"
]

print(f"{'Table':<25} {'Rows':>10}  Status")
print("-" * 50)

for table_name in tables_to_ingest:
    try:
        df = spark.read.jdbc(
            url        = jdbc_url,
            table      = table_name,
            properties = connection_properties
        )
        row_count = df.count()

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{GBMART_CATALOG}.bronze.{table_name}")

        print(f"{table_name:<25} {row_count:>10,}  Saved to gbmart.bronze from Supabase")

    except Exception as e:
        print(f"{table_name:<25} {'':>10}  ERROR: {str(e)[:60]}")

print("-" * 50)
print("When this works, you no longer need CSV files — data comes direct from Supabase!")

## Section 5 — Enabling Logical Replication in Supabase (prep for HOL 2)

> This is what **Lakeflow Connect's log-based mode** does automatically for a pipeline attached to a Postgres connection — not the mode GlobalMart's real `orders`/`order_items` pipeline uses (that one is query/cursor-based, see ILT 2). Doing this by hand here — and again in **HOL 2** — teaches you the log-based mechanism as general CDC knowledge: what a replication slot is and why it matters, even on a pipeline that doesn't use one.

### Steps in Supabase Dashboard

```
1. Log in to Supabase → your project

2. Go to: Database → Replication

3. Under "Source" — you will see your tables listed

4. Toggle ON the tables you want CDC for:
   ✅ customers      (address + email changes)
   ✅ orders         (status changes: pending → shipped → delivered)
   ✅ addresses      (customers move)

5. This sets: wal_level = logical in PostgreSQL config
   Supabase handles the rest automatically
```

### What happens after enabling

```
Someone updates a customer's city in the app
    ↓
PostgreSQL writes the change to WAL
    ↓
Logical replication decodes it:
    {"op": "UPDATE", "table": "customers", "id": "C001",
     "city": {"old": "Mumbai", "new": "Delhi"}}
    ↓
We read this change event manually via JDBC in HOL 2 —
this is log-based CDC, the mode GlobalMart's real pipeline does NOT use
    ↓
MERGE into Silver → Gold reflects the new city
```

> **In HOL 2:** You will make a change in your own Supabase project (update/insert/delete a row on `poc_orders`), then read the raw WAL change events back in Databricks via a replication slot — log-based CDC, worth knowing hands-on even though GlobalMart's real `orders`/`order_items` pipeline is configured in the other (query/cursor) mode instead.

## Recap

| Topic | Key Takeaway |
|-------|--------------|
| Ingestion | Bringing data from a source into Bronze — every pattern picks Full/Incremental and Batch/Streaming |
| Common ingestion issues | Schema evolution, duplicates, late-arriving data, format mismatches — handled in Silver, not Bronze |
| Why CDC | Incremental load misses UPDATEs and DELETEs — CDC captures all 3 |
| WAL | PostgreSQL's change log — the source of log-based CDC |
| Logical replication | Decodes WAL into readable INSERT/UPDATE/DELETE events |
| JDBC | Direct database connection — reads tables from Supabase into Spark (teaching pattern) |
| JDBC + `updated_at` | Simple CDC pattern — reads rows changed since last run (misses deletes) — same idea GlobalMart's real pipeline uses via Lakeflow Connect |
| Lakeflow Connect | Databricks' managed CDC tool — can run log-based (WAL) or query/cursor-based. **GlobalMart's real pipeline uses query/cursor mode** — captures inserts/updates, not deletes |
| Supabase setup | Enable logical replication in Dashboard → Database → Replication (needed for the log-based mode this session teaches by hand, not for GlobalMart's actual pipeline) |

---

## What Comes Next

| Session | Topic |
|---------|-------|
| **ILT 2 (next)** | Lakeflow Connect — configuring GlobalMart's real query/cursor-based CDC pipeline + why/how to create Storage Credentials & External Locations in Unity Catalog |
| **HOL 1** | Hands-on — create your own Storage Credential + External Location, and set up a Lakeflow Connect pipeline against your own Supabase `orders`/`order_items` |
| **HOL 2** | Hands-on — the WAL/replication-slot mechanics directly via JDBC (log-based CDC, the mode Lakeflow Connect *could* run but GlobalMart's real pipeline doesn't) |
| **ILT 3 / HOL 3** | Code Versioning — Databricks Repos + GitHub |

---

**INSTRUCTOR NOTE:**
Closing check:
1. *'What are the two decisions every ingestion pattern makes?'* (How much data to bring — Full vs Incremental — and how continuously — Batch vs Streaming.)
2. *'Why doesn't a watermark filter catch deletes?'* (The row is gone — there's no `updated_at` to filter on anymore.)
3. *'What does a PostgreSQL replication slot do?'* (Bookmarks the WAL position so you can resume reading changes without gaps or duplicates — this is the log-based mode, not what GlobalMart's real pipeline uses.)
4. *'What is Lakeflow Connect, in one sentence?'* (A managed Databricks pipeline that can read a Postgres WAL via a replication slot, or re-query on a cursor column — GlobalMart's real pipeline uses the cursor column mode, `updated_at`, and does not capture hard deletes.)